In [199]:
from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Annotated
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
from langchain_groq import ChatGroq

In [200]:
load_dotenv()

True

In [201]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [202]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedback for the essay')
    score: float = Field(description='Score out of 10', ge=0, le=10)

In [203]:
structured_model = model.with_structured_output(
    EvaluationSchema,
)

In [204]:
essay = """
    The software-development job market is changing rapidly. The traditional profile of a developer—someone who primarily writes frontend or backend code—is increasingly evolving toward a broader role in which developers are expected to understand application architecture, cloud infrastructure, databases, AI systems, and AI-assisted development. In this environment, combining Full-Stack Web Development with TypeScript, Agentic AI development, DSA, and SQL can create a particularly strong profile.

This combination is valuable because it brings together two complementary abilities: building reliable software systems and integrating intelligent behavior into those systems.
 """

In [205]:
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback:  str
    overall_feedback: str
    individual_scores: Annotated[list[float], operator.add]
    avg_score : float

In [206]:
def evaluate_language(state: EssayState):

    prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)

    return {
        'language_feedback' : output.feedback,
        'individual_scores' : [output.score]
    }

In [207]:
def evaluate_analysis(state: EssayState):

    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)

    return {
        'analysis_feedback' : output.feedback,
        'individual_scores' : [output.score]
    }

In [208]:
def evaluate_thought(state: EssayState):

    prompt = f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)

    return {
        'clarity_feedback' : output.feedback,
        'individual_scores' : [output.score]
    }

In [216]:
def final_evaluation(state: EssayState):
    prompt = (
        "Based on the following feedbacks create an overall summarized feedback\n"
        f"language feedback - {state['language_feedback']}\n"
        f"analysis feedback - {state['analysis_feedback']}\n"
        f"clarity feedback - {state['clarity_feedback']}"
    )
    overall_feedback = model.invoke(prompt).content

    scores = state.get("individual_scores", [])
    avg_score = sum(scores) / len(scores) if scores else 0.0

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }

In [210]:
graph = StateGraph(EssayState)

# add nodes
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# add edges 
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought' , 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [211]:
essay2 = """
The 2026 FIFA World Cup was a historic global football tournament jointly hosted by the United States, Canada, and Mexico from June 11 to July 19, 2026.
Introduction
The 2026 FIFA World Cup marked the 23rd edition of the quadrennial international men's football championship. It broke tradition as the first tournament to be co-hosted by three different nations and the largest edition in football history.
A New Format and Expansion
For the first time, the tournament expanded from 32 teams to a record 48 national teams.
• The teams were split into 12 groups of four.
• A new Round of 32 knockout stage was added.
• The total match count grew from 64 to 104 matches spread across 16 host cities (11 in the U.S., 3 in Mexico, and 2 in Canada).
• This expansion allowed smaller football nations and debutants greater access to the global stage.
Tournament Highlights and the Final
The 39-day event captivated billions of fans worldwide with high-intensity action. The grand final took place at the New York New Jersey Stadium. In a thrilling climax, Spain defeated Argentina 1–0 in extra time, with Mikel Oyarzabal scoring the decisive goal in the 63rd minute to claim Spain's second World Cup title. Kylian Mbappé won the Golden Boot with 10 goals, while Spanish midfielder Rodri was awarded the Golden Ball.
Conclusion
The 2026 FIFA World Cup redefined the scale of international sports. By uniting three North American host nations and embracing a larger 48-team format, the tournament successfully globalized the game while delivering an unforgettable month of athletic excellence.
"""

In [217]:
initial_state = {
    'essay' : essay2
}

# Rebuild the graph with the corrected node
graph = StateGraph(EssayState)

graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")
graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")
graph.add_edge("final_evaluation", END)

workflow = graph.compile()

result = workflow.invoke(initial_state)
print(result)

{'essay': "\nThe 2026 FIFA World Cup was a historic global football tournament jointly hosted by the United States, Canada, and Mexico from June 11 to July 19, 2026.\nIntroduction\nThe 2026 FIFA World Cup marked the 23rd edition of the quadrennial international men's football championship. It broke tradition as the first tournament to be co-hosted by three different nations and the largest edition in football history.\nA New Format and Expansion\nFor the first time, the tournament expanded from 32 teams to a record 48 national teams.\n• The teams were split into 12 groups of four.\n• A new Round of 32 knockout stage was added.\n• The total match count grew from 64 to 104 matches spread across 16 host cities (11 in the U.S., 3 in Mexico, and 2 in Canada).\n• This expansion allowed smaller football nations and debutants greater access to the global stage.\nTournament Highlights and the Final\nThe 39-day event captivated billions of fans worldwide with high-intensity action. The grand fin